# 🦥 Itemset Extractor — SFT → DPO Fine-Tuning (Unsloth)

**Model:** `Qwen/Qwen2.5-3B-Instruct`  
**Method:** SFT warmup (1 epoch) → DPO fine-tuning (2 epochs)  
**Library:** Unsloth — 2× faster, 70% less VRAM  
**Dataset:** `OliverSlivka/itemset-extraction-rlhf-v1`  

## Why SFT → DPO?

| Phase | Purpose | Loss | Duration |
|-------|---------|------|----------|
| **SFT** (Phase 1) | Teach the task format — correct JSON itemset structure | Cross-entropy on `chosen` responses | ~10–15 min |
| **DPO** (Phase 2) | Refine using preferences — penalise hallucinations, wrong counts, missing itemsets | Preference ranking loss on `(chosen, rejected)` pairs | ~20–30 min |

Pure DPO from scratch on synthetic rejected samples is suboptimal. SFT first ensures the model understands the output format before preference learning starts.

## Bonus: GRPO (next iteration, cells at the bottom)
When you have enough training data, GRPO with an Apriori F1 reward function is the most optimal approach — no synthetic rejected samples needed, the reward is fully programmatic.

In [ ]:
# ── Cell 2: Install ──────────────────────────────────────────────────────────
# Run once on first launch. Restart kernel after installing.
%pip install unsloth trl datasets transformers accelerate bitsandbytes huggingface_hub -q

In [ ]:
# ── Cell 3: CONFIG — edit this cell only ─────────────────────────────────────
CONFIG = {
    # ── Model ────────────────────────────────────────────────────────────────
    "base_model":        "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",  # pre-quantized: 4× faster download
    "max_seq_length":    2048,
    "load_in_4bit":      True,

    # ── Dataset (HuggingFace Hub) ─────────────────────────────────────────────
    "hf_dataset":        "OliverSlivka/itemset-extraction-rlhf-v1",
    "hf_token":          "",    # paste your HF read token here (or set env HF_TOKEN)

    # ── LoRA (shared across SFT + DPO) ───────────────────────────────────────
    "lora_r":            64,
    "lora_alpha":        16,    # rule of thumb: lora_alpha = lora_r / 4
    "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                             "gate_proj", "up_proj", "down_proj"],

    # ── Phase 1: SFT ─────────────────────────────────────────────────────────
    "sft_epochs":        1,     # 1 epoch is enough to learn the format
    "sft_lr":            2e-4,
    "sft_batch_size":    2,
    "sft_grad_accum":    8,     # effective batch = 16
    "sft_output_dir":    "./sft_checkpoint",

    # ── Phase 2: DPO ─────────────────────────────────────────────────────────
    "dpo_epochs":        2,
    "dpo_lr":            5e-5,  # always lower than SFT lr
    "dpo_beta":          0.1,   # preference strength; 0.1 = balanced
    "dpo_batch_size":    1,
    "dpo_grad_accum":    8,
    "dpo_output_dir":    "./dpo_final",

    # ── Output / Push ─────────────────────────────────────────────────────────
    "hf_model_repo":     "OliverSlivka/qwen2.5-3b-itemset-extractor",
    "push_to_hub":       True,
}

print("✅ CONFIG loaded")

In [ ]:
# ── Cell 4: GPU check + imports ───────────────────────────────────────────────
import os, gc, json, re, torch
from datasets import load_dataset
from huggingface_hub import login

# HF login
hf_token = CONFIG["hf_token"] or os.environ.get("HF_TOKEN", "")
if hf_token:
    login(token=hf_token)
    print("✅ HuggingFace logged in")
else:
    print("⚠️  No HF token — dataset load will fail if repo is private")

# GPU info
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM total : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   VRAM free  : {torch.cuda.memory_reserved(0) / 1e9:.1f} GB reserved")
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
else:
    raise RuntimeError("❌ No GPU found — connect a GPU runtime before continuing.")

In [ ]:
# ── Cell 5: Load dataset ──────────────────────────────────────────────────────
raw_dataset = load_dataset(CONFIG["hf_dataset"])
print(f"Train : {len(raw_dataset['train'])} examples")
print(f"Val   : {len(raw_dataset['validation'])} examples")
print("\nExample keys:", list(raw_dataset["train"][0].keys()))
print("\nSample prompt (first message):", raw_dataset["train"][0]["prompt"][0]["content"][:200], "...")

## Phase 1 — SFT Warmup 🎓

Trains on the **chosen** responses only (Apriori ground truth).  
Goal: teach the model *what the correct output format looks like* before any preference learning.  
- 1 epoch, lr = 2e-4
- Loss: standard cross-entropy (next-token prediction on full conversation)

In [ ]:
# ── Cell 7: Load model for SFT (Unsloth) ─────────────────────────────────────
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = CONFIG["base_model"],
    max_seq_length= CONFIG["max_seq_length"],
    load_in_4bit  = CONFIG["load_in_4bit"],
    dtype         = None,  # auto: bfloat16 on Ampere+, float16 on older
)

model = FastLanguageModel.get_peft_model(
    model,
    r                        = CONFIG["lora_r"],
    lora_alpha               = CONFIG["lora_alpha"],
    target_modules           = CONFIG["lora_target_modules"],
    lora_dropout             = 0,       # 0 = Unsloth-optimized path
    bias                     = "none",
    use_gradient_checkpointing= "unsloth",  # saves ~30% extra VRAM on top of 4-bit
    random_state             = 42,
)
model.print_trainable_parameters()

In [ ]:
# ── Cell 8: Prepare SFT dataset + train ──────────────────────────────────────
from trl import SFTTrainer, SFTConfig

def format_sft(examples):
    """Convert RLHF pairs → plain conversation text for SFT.
    Uses only the `chosen` (Apriori ground truth) response.
    """
    texts = []
    for prompt, chosen in zip(examples["prompt"], examples["chosen"]):
        # prompt = [system_msg, user_msg], chosen = [assistant_msg]
        messages = prompt + chosen
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

sft_train = raw_dataset["train"].map(format_sft, batched=True,
                                      remove_columns=raw_dataset["train"].column_names)
sft_val   = raw_dataset["validation"].map(format_sft, batched=True,
                                           remove_columns=raw_dataset["validation"].column_names)

sft_trainer = SFTTrainer(
    model            = model,
    tokenizer        = tokenizer,
    train_dataset    = sft_train,
    eval_dataset     = sft_val,
    args             = SFTConfig(
        dataset_text_field         = "text",
        max_seq_length             = CONFIG["max_seq_length"],
        num_train_epochs           = CONFIG["sft_epochs"],
        per_device_train_batch_size= CONFIG["sft_batch_size"],
        gradient_accumulation_steps= CONFIG["sft_grad_accum"],
        learning_rate              = CONFIG["sft_lr"],
        lr_scheduler_type          = "cosine",
        warmup_ratio               = 0.05,
        optim                      = "adamw_8bit",
        bf16                       = True,
        fp16                       = False,
        logging_steps              = 10,
        eval_strategy              = "epoch",
        save_strategy              = "epoch",
        output_dir                 = CONFIG["sft_output_dir"],
        report_to                  = "none",
    ),
)

print("🎓 Starting SFT warmup...")
sft_trainer.train()
print("✅ SFT done")

In [ ]:
# ── Cell 9: Save SFT checkpoint (merged 4-bit) ────────────────────────────────
# Merges LoRA weights into base model and saves as a quantized 4-bit model.
# This lets DPO load it as a fresh base — cleaner than continuing with the same adapters.
model.save_pretrained_merged(
    CONFIG["sft_output_dir"],
    tokenizer,
    save_method = "merged_4bit_forced",
)
print(f"✅ SFT checkpoint saved → {CONFIG['sft_output_dir']}")

# Free memory before loading fresh for DPO
del model
gc.collect()
torch.cuda.empty_cache()
print(f"   VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.1f} GB")

## Phase 2 — DPO Fine-Tuning 🎯

Loads the SFT checkpoint and trains with preference pairs:  
- **chosen** = Apriori ground truth (correct itemsets)  
- **rejected** = synthetic errors (hallucinations, missing itemsets, wrong counts, etc.)

Beta = 0.1 (balanced). Higher → stronger preference signal, higher overfitting risk.

In [ ]:
# ── Cell 11: Reload SFT checkpoint + apply fresh LoRA for DPO ────────────────
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = CONFIG["sft_output_dir"],  # load from SFT checkpoint
    max_seq_length= CONFIG["max_seq_length"],
    load_in_4bit  = CONFIG["load_in_4bit"],
    dtype         = None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                        = CONFIG["lora_r"],
    lora_alpha               = CONFIG["lora_alpha"],
    target_modules           = CONFIG["lora_target_modules"],
    lora_dropout             = 0,
    bias                     = "none",
    use_gradient_checkpointing= "unsloth",
    random_state             = 42,
)
model.print_trainable_parameters()
print("✅ SFT checkpoint loaded for DPO")

In [ ]:
# ── Cell 12: Prepare DPO dataset + train ─────────────────────────────────────
from trl import DPOTrainer, DPOConfig

def format_dpo(examples):
    """Convert RLHF pairs → DPO format: (prompt_str, chosen_str, rejected_str)."""
    prompts, chosens, rejecteds = [], [], []
    for prompt, chosen, rejected in zip(examples["prompt"], examples["chosen"], examples["rejected"]):
        prompts.append(
            tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
        )
        chosens.append(chosen[0]["content"])
        rejecteds.append(rejected[0]["content"])
    return {"prompt": prompts, "chosen": chosens, "rejected": rejecteds}

dpo_train = raw_dataset["train"].map(format_dpo, batched=True,
                                      remove_columns=raw_dataset["train"].column_names)
dpo_val   = raw_dataset["validation"].map(format_dpo, batched=True,
                                           remove_columns=raw_dataset["validation"].column_names)

dpo_trainer = DPOTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = dpo_train,
    eval_dataset     = dpo_val,
    args             = DPOConfig(
        beta                       = CONFIG["dpo_beta"],
        max_length                 = CONFIG["max_seq_length"],
        max_prompt_length          = CONFIG["max_seq_length"] // 2,
        num_train_epochs           = CONFIG["dpo_epochs"],
        per_device_train_batch_size= CONFIG["dpo_batch_size"],
        gradient_accumulation_steps= CONFIG["dpo_grad_accum"],
        learning_rate              = CONFIG["dpo_lr"],
        lr_scheduler_type          = "cosine",
        warmup_ratio               = 0.05,
        optim                      = "adamw_8bit",
        bf16                       = True,
        fp16                       = False,
        gradient_checkpointing     = False,  # Unsloth handles its own
        logging_steps              = 10,
        eval_strategy              = "epoch",
        save_strategy              = "epoch",
        save_total_limit           = 2,
        load_best_model_at_end     = True,
        output_dir                 = CONFIG["dpo_output_dir"],
        report_to                  = "none",
    ),
)

print("🎯 Starting DPO fine-tuning...")
dpo_trainer.train()
print("✅ DPO done")

In [ ]:
# ── Cell 13: Save final model + push to HuggingFace Hub ──────────────────────
import os

# Save merged 4-bit model locally
model.save_pretrained_merged(
    CONFIG["dpo_output_dir"] + "/final",
    tokenizer,
    save_method = "merged_4bit_forced",
)
print(f"✅ Final model saved → {CONFIG['dpo_output_dir']}/final")

# Push to HF Hub
if CONFIG["push_to_hub"]:
    model.push_to_hub_merged(
        CONFIG["hf_model_repo"],
        tokenizer,
        save_method = "merged_4bit_forced",
        token       = CONFIG["hf_token"] or os.environ.get("HF_TOKEN", ""),
    )
    tokenizer.push_to_hub(CONFIG["hf_model_repo"],
                          token=CONFIG["hf_token"] or os.environ.get("HF_TOKEN", ""))
    print(f"🚀 Pushed to HF Hub → https://huggingface.co/{CONFIG['hf_model_repo']}")
else:
    print("ℹ️  push_to_hub=False — model only saved locally")

In [ ]:
# ── Cell 14: Quick sanity-check inference ────────────────────────────────────
# Run a small test to confirm the model produces valid JSON itemsets.
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)  # enable native 2x inference

SAMPLE_CSV = """Transaction,Items
Row 1,milk bread eggs
Row 2,milk bread
Row 3,bread eggs butter
Row 4,milk eggs
Row 5,milk bread eggs butter"""

messages = [
    {"role": "system", "content": "You are an expert in frequent itemset mining. "
                                   "Extract all frequent itemsets with min_support=3 from the transaction data. "
                                   "Return a JSON array with keys: itemset, count, support, rows."},
    {"role": "user", "content": f"Find frequent itemsets (min_support=3) in this data:\n\n{SAMPLE_CSV}"},
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids        = inputs,
    max_new_tokens   = 512,
    temperature      = 0.1,
    do_sample        = True,
    pad_token_id     = tokenizer.eos_token_id,
)

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("─── Model Output ───")
print(response)

# Validate JSON
try:
    parsed = json.loads(re.search(r'\[.*\]', response, re.DOTALL).group())
    print(f"\n✅ Valid JSON — {len(parsed)} itemsets found")
    for item in parsed:
        print(f"  {item.get('itemset')}  count={item.get('count')}  support={item.get('support'):.2f}")
except Exception as e:
    print(f"\n⚠️  JSON parse failed: {e}")

---
## 🔬 Advanced: GRPO with Apriori Reward (Next Iteration)

This is the **most optimal** approach for this task once you have enough training data:
- No synthetic rejected samples needed
- The reward function is **fully programmatic** — it computes F1 against Apriori ground truth
- The model learns from its own actual failures, not pre-fabricated ones

**When to switch to GRPO:** After SFT → DPO gives you a strong baseline F1 (> 0.70), GRPO can push further.

Run cells 15–16 **instead of** cells 8–13 on your next training iteration.

In [ ]:
# ── Cell 15: GRPO reward functions ───────────────────────────────────────────
# These reward functions use Apriori ground truth — no synthetic data needed.

def _parse_itemsets(text):
    """Parse model output → list of itemset dicts. Returns None on failure."""
    try:
        m = re.search(r'\[.*\]', text, re.DOTALL)
        if m:
            return json.loads(m.group())
    except Exception:
        pass
    return None


def json_format_reward(completions, **kwargs):
    """Reward: is the output a valid JSON array with 'itemset' keys? (0.0–1.0)"""
    rewards = []
    for comp in completions:
        text = comp[0]["content"] if isinstance(comp, list) else comp
        parsed = _parse_itemsets(text)
        if parsed is None:
            rewards.append(0.0)
        elif not isinstance(parsed, list) or len(parsed) == 0:
            rewards.append(0.2)
        elif all("itemset" in item for item in parsed):
            rewards.append(1.0)
        else:
            rewards.append(0.5)
    return rewards


def itemset_f1_reward(completions, chosen, **kwargs):
    """Reward: F1 score of predicted itemsets vs Apriori ground truth.
    `chosen` column from the DPO dataset holds the ground truth response.
    """
    rewards = []
    for comp, gt_response in zip(completions, chosen):
        text = comp[0]["content"] if isinstance(comp, list) else comp
        gt_text = gt_response[0]["content"] if isinstance(gt_response, list) else gt_response

        predicted = _parse_itemsets(text)
        ground_truth = _parse_itemsets(gt_text)

        if predicted is None or ground_truth is None:
            rewards.append(0.0)
            continue

        pred_sets = {frozenset(item["itemset"]) for item in predicted if "itemset" in item}
        true_sets = {frozenset(item["itemset"]) for item in ground_truth if "itemset" in item}

        if not true_sets:
            rewards.append(1.0 if not pred_sets else 0.0)
            continue

        tp = len(pred_sets & true_sets)
        precision = tp / len(pred_sets) if pred_sets else 0.0
        recall    = tp / len(true_sets)
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        rewards.append(f1)
    return rewards


def combined_reward(completions, chosen, **kwargs):
    """Combined: 0.3 × format_reward + 0.7 × F1_reward"""
    fmt = json_format_reward(completions)
    f1  = itemset_f1_reward(completions, chosen)
    return [0.3 * f + 0.7 * fi for f, fi in zip(fmt, f1)]


print("✅ GRPO reward functions defined")

In [ ]:
# ── Cell 16: GRPO training (run after SFT, instead of DPO) ───────────────────
# Prereqs: run cells 7–9 (SFT warmup + save checkpoint) first,
#          then reload from SFT checkpoint (cell 11), then run THIS cell instead of cell 12.
#
# from trl import GRPOTrainer, GRPOConfig
#
# grpo_dataset = raw_dataset["train"].map(
#     lambda ex: {
#         "prompt": tokenizer.apply_chat_template(
#             ex["prompt"], tokenize=False, add_generation_prompt=True
#         ),
#         "chosen": ex["chosen"],  # passed as kwarg to reward fn
#     }
# )
#
# grpo_trainer = GRPOTrainer(
#     model         = model,
#     reward_funcs  = [combined_reward],   # Apriori F1 reward
#     processing_class = tokenizer,
#     train_dataset = grpo_dataset,
#     args          = GRPOConfig(
#         num_train_epochs           = 2,
#         per_device_train_batch_size= 1,
#         gradient_accumulation_steps= 8,
#         learning_rate              = 1e-5,   # lower than DPO
#         max_completion_length      = 512,
#         num_generations            = 4,      # responses per prompt (GRPO sampling)
#         temperature                = 0.9,
#         bf16                       = True,
#         output_dir                 = "./grpo_final",
#         report_to                  = "none",
#     ),
# )
#
# print("🔬 Starting GRPO training...")
# grpo_trainer.train()
# print("✅ GRPO done")

print("ℹ️  GRPO cell is commented out — uncomment when ready for next iteration")